# Plot Mini: train the Plotlines command model

Fine-tunes a small model (Gemma 3 270M by default) with LoRA to turn a spoken request into the one JSON action the app's
`llmIntent` expects, then exports it for the phone:

- `plot-mini-q4_0.gguf` for the native llama.cpp engine (the default in the Android app)
- `plot-mini.litertlm` for the GPU path (LiteRT-LM)

Runtime: Colab with a T4 GPU is enough (about 10 minutes for 270M, 25 for 1B). Run the cells top to bottom.

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes

## 1. Data
The generator reads the prompt and the action list from `assistant.js`, so the model learns the exact prompt the app sends.

In [ ]:
!git clone --depth 1 https://github.com/Munazzar/plotline
!python plotline/train/gen_dataset.py -n 8000
from datasets import load_dataset
ds = load_dataset('json', data_files={'train': 'plotline/train/data/train.jsonl', 'val': 'plotline/train/data/val.jsonl'})
ds

## 2. Model
Pick one. 270M answers in well under a second on a phone; 1B and Qwen 0.5B are a little smarter with odd phrasing.

In [ ]:
BASE = 'unsloth/gemma-3-270m-it'      # or 'unsloth/gemma-3-1b-it', 'unsloth/Qwen2.5-0.5B-Instruct'
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(BASE, max_seq_length=1024, load_in_4bit=False, full_finetuning=False)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'], use_gradient_checkpointing='unsloth', random_state=7)

In [ ]:
def fmt(b):
    return {'text': [tokenizer.apply_chat_template(m, tokenize=False) for m in b['messages']]}
train = ds['train'].map(fmt, batched=True, remove_columns=['messages'])

## 3. Train (loss only on the JSON answer)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train,
    args=SFTConfig(dataset_text_field='text', per_device_train_batch_size=16, gradient_accumulation_steps=1, num_train_epochs=2,
                   learning_rate=2e-4, warmup_steps=20, lr_scheduler_type='cosine', logging_steps=25, optim='adamw_8bit',
                   weight_decay=0.01, seed=7, output_dir='out', report_to='none'))
# Gemma turn markers; for Qwen use '<|im_start|>user
' and '<|im_start|>assistant
'
if 'gemma' in BASE:
    trainer = train_on_responses_only(trainer, instruction_part='<start_of_turn>user
', response_part='<start_of_turn>model
')
else:
    trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user
', response_part='<|im_start|>assistant
')
trainer.train()

## 4. Check on held-out requests
The action must match, and the text must be what gets saved (no command words).

In [ ]:
import json, torch
FastLanguageModel.for_inference(model)
ok = okt = 0
val = ds['val'].select(range(200))
for ex in val:
    m = ex['messages']
    ids = tokenizer.apply_chat_template(m[:2], add_generation_prompt=True, return_tensors='pt').to('cuda')
    out = model.generate(input_ids=ids, max_new_tokens=80, do_sample=False)
    s = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    want = json.loads(m[2]['content'])
    try: got = json.loads(s[s.index('{'):s.rindex('}')+1])
    except Exception: got = {}
    ok += got.get('kind') == want['kind']; okt += got == want
print(f'kind right: {ok/len(val):.1%}   whole answer right: {okt/len(val):.1%}')

## 5. Export GGUF Q4_0 (native llama.cpp in the Android app)

In [ ]:
model.save_pretrained_gguf('plot-mini', tokenizer, quantization_method='q4_0')
import glob, shutil
src = [f for f in glob.glob('plot-mini*/*.gguf') + glob.glob('plot-mini*.gguf') if 'q4_0' in f.lower()][0]
shutil.copy(src, 'plot-mini-q4_0.gguf'); print(src, '->', 'plot-mini-q4_0.gguf')

## 6. Export `.litertlm` (GPU path, optional)

Merges the LoRA into the base weights, converts with Google's LiteRT Torch converter and bundles the tokenizer into a
`.litertlm`. The converter moves fast: if a module path below has changed, check the `litert-torch` / `ai-edge-torch`
README for the Gemma 3 example. The GGUF from step 5 is all the app needs by default.

In [ ]:
model.save_pretrained_merged('merged', tokenizer, save_method='merged_16bit')
!pip install -q ai-edge-torch-nightly litert-lm
!python -m ai_edge_torch.generative.examples.gemma3.convert_gemma3_to_tflite     --checkpoint_path merged --model_size 270m --output_path litert --output_name_prefix plot-mini     --prefill_seq_lens 512 --kv_cache_max_len 1024 --quantize dynamic_int8
!python -m litert_lm.builder --tflite_model litert/plot-mini*.tflite --tokenizer merged/tokenizer.model     --llm_model_type gemma3 --output plot-mini.litertlm

## 7. Upload
The app lists Plot Mini once these two files are at `huggingface.co/Munazzar/plot-mini` (exact names).

In [ ]:
from huggingface_hub import login, HfApi
login()  # a write token from huggingface.co/settings/tokens
api = HfApi(); api.create_repo('Munazzar/plot-mini', exist_ok=True)
for f in ['plot-mini-q4_0.gguf', 'plot-mini.litertlm']:
    try: api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id='Munazzar/plot-mini'); print('uploaded', f)
    except FileNotFoundError: print('skipped', f)